# 📊 Exploratory Data Analysis (EDA)
## Enron Spam Dataset
**Goal**: Understand the data before preprocessing

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
# Load raw data
df = pd.read_csv('../data/raw/enron_spam_data.csv')
print(f"✅ Dataset loaded: {len(df)} emails")
df.head()

In [ ]:
# Basic info
df.info()

## 2. Data Overview

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
# Check class distribution
df['Spam/Ham'].value_counts()

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
df['Spam/Ham'].value_counts().plot.pie(
    ax=axes[0], autopct='%1.1f%%', explode=[0, 0.05],
    colors=['#2ecc71', '#e74c3c'], shadow=True
)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('')

# Bar chart
df['Spam/Ham'].value_counts().plot.bar(
    ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='black'
)
axes[1].set_title('Class Counts')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')

for i, v in enumerate(df['Spam/Ham'].value_counts().values):
    axes[1].text(i, v + 200, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Text Analysis
- Length distribution
- Word count distribution
- Common words in spam vs ham

In [ ]:
# Add text length columns
df['text_length'] = df['Message'].fillna('').apply(len)
df['subject_length'] = df['Subject'].fillna('').apply(len)
df['word_count'] = df['Message'].fillna('').apply(lambda x: len(str(x).split()))

# Summary statistics
df[['text_length', 'subject_length', 'word_count']].describe()

In [ ]:
# Distribution of text length by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
df.boxplot(column='text_length', by='Spam/Ham', ax=axes[0])
axes[0].set_title('Text Length Distribution by Class')
axes[0].set_ylabel('Text Length')

# Histogram
for label, color in [('ham', '#2ecc71'), ('spam', '#e74c3c')]:
    subset = df[df['Spam/Ham'] == label]['text_length']
    subset.hist(bins=50, alpha=0.5, label=label.capitalize(), color=color, ax=axes[1])
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Text Length')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Most Common Words

In [ ]:
from collections import Counter
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def get_common_words(texts, n=20):
    words = []
    for text in texts:
        if isinstance(text, str):
            text = text.lower()
            text = re.sub(r'[^a-zA-Z\s]', '', text)
            words.extend([w for w in text.split() if w not in stop_words and len(w) > 2])
    return Counter(words).most_common(n)

# Common words in spam
spam_texts = df[df['Spam/Ham'] == 'spam']['Message'].fillna('')
spam_words = get_common_words(spam_texts, 20)

# Common words in ham
ham_texts = df[df['Spam/Ham'] == 'ham']['Message'].fillna('')
ham_words = get_common_words(ham_texts, 20)

In [ ]:
# Plot common words
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Spam words
words_spam, counts_spam = zip(*spam_words)
axes[0].barh(words_spam, counts_spam, color='#e74c3c', edgecolor='black')
axes[0].set_title('Top 20 Words in SPAM')
axes[0].set_xlabel('Frequency')
axes[0].invert_yaxis()

# Ham words
words_ham, counts_ham = zip(*ham_words)
axes[1].barh(words_ham, counts_ham, color='#2ecc71', edgecolor='black')
axes[1].set_title('Top 20 Words in HAM')
axes[1].set_xlabel('Frequency')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/figures/common_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Subject Line Analysis

In [ ]:
# Check if subject is empty
df['has_subject'] = df['Subject'].notna() & (df['Subject'] != '')

subject_stats = df.groupby('Spam/Ham')['has_subject'].value_counts(normalize=True).unstack()
subject_stats

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
subject_stats.plot(kind='bar', ax=ax, color=['#95a5a6', '#34495e'], edgecolor='black')
ax.set_title('Percentage of Emails with Subject')
ax.set_xlabel('Class')
ax.set_ylabel('Proportion')
ax.legend(['No Subject', 'Has Subject'])
ax.set_xticklabels(['Ham', 'Spam'], rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/subject_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Special Characters Analysis

In [ ]:
# Count special characters
df['exclamation'] = df['Message'].fillna('').apply(lambda x: str(x).count('!'))
df['question'] = df['Message'].fillna('').apply(lambda x: str(x).count('?'))
df['dollar'] = df['Message'].fillna('').apply(lambda x: str(x).count('$'))
df['percent'] = df['Message'].fillna('').apply(lambda x: str(x).count('%'))

# Summary by class
df.groupby('Spam/Ham')[['exclamation', 'question', 'dollar', 'percent']].mean()

In [ ]:
# Visualize special characters
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
chars = ['exclamation', 'question', 'dollar', 'percent']
titles = ['Exclamation Marks (!)', 'Question Marks (?)', 'Dollar Signs ($)', 'Percent Signs (%)']

for i, (char, title) in enumerate(zip(chars, titles)):
    row, col = i // 2, i % 2
    df.boxplot(column=char, by='Spam/Ham', ax=axes[row, col])
    axes[row, col].set_title(title)
    axes[row, col].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../reports/figures/special_characters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis
Understanding relationships between features

In [ ]:
# Convert Spam/Ham to numeric for correlation
df['label_num'] = df['Spam/Ham'].map({'ham': 0, 'spam': 1})

# Select numeric columns
numeric_cols = ['text_length', 'subject_length', 'word_count', 'exclamation', 
                'question', 'dollar', 'percent', 'label_num']

# Correlation matrix
corr = df[numeric_cols].corr()

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Insights Summary

In [ ]:
print("="*60)
print("📊 KEY INSIGHTS FROM EDA")
print("="*60)

print(f"\n1. Dataset Size: {len(df)} emails")
print(f"   - Ham: {len(df[df['Spam/Ham']=='ham'])} ({len(df[df['Spam/Ham']=='ham'])/len(df)*100:.1f}%)")
print(f"   - Spam: {len(df[df['Spam/Ham']=='spam'])} ({len(df[df['Spam/Ham']=='spam'])/len(df)*100:.1f}%)")

print(f"\n2. Text Length:")
print(f"   - Average Ham Length: {df[df['Spam/Ham']=='ham']['text_length'].mean():.0f} chars")
print(f"   - Average Spam Length: {df[df['Spam/Ham']=='spam']['text_length'].mean():.0f} chars")

print(f"\n3. Subject Lines:")
ham_subject = df[(df['Spam/Ham']=='ham') & (df['has_subject']==True)].shape[0]
spam_subject = df[(df['Spam/Ham']=='spam') & (df['has_subject']==True)].shape[0]
print(f"   - Ham with Subject: {ham_subject} ({ham_subject/len(df[df['Spam/Ham']=='ham'])*100:.1f}%)")
print(f"   - Spam with Subject: {spam_subject} ({spam_subject/len(df[df['Spam/Ham']=='spam'])*100:.1f}%)")

print(f"\n4. Special Characters (Average):")
print(f"   - Exclamation: Ham={df[df['Spam/Ham']=='ham']['exclamation'].mean():.2f}, Spam={df[df['Spam/Ham']=='spam']['exclamation'].mean():.2f}")
print(f"   - Dollar Signs: Ham={df[df['Spam/Ham']=='ham']['dollar'].mean():.2f}, Spam={df[df['Spam/Ham']=='spam']['dollar'].mean():.2f}")

print(f"\n5. Top Spam Words: {', '.join([w for w, c in spam_words[:5]])}")
print(f"6. Top Ham Words: {', '.join([w for w, c in ham_words[:5]])}")

print("\n" + "="*60)
print("✅ EDA COMPLETE! Ready for preprocessing.")
print("="*60)